# Содержание

* Бизнес–контекст (как его формулировали разработчики)
* Импорт библиотек и чтение данных
* Обработка данных
* Проверка соотношения групп (SRM)
* Тест гипотез
  * Анализ удержания на 7-й день
  * Анализ удержания на 1-й день
  * Анализ количества раундов
* Выводы и рекомендации
* «Как бы я спроектировал тест, если бы запускал сам»

# Бизнес–контекст (как его формулировали разработчики)

Cookie Cats — популярная мобильная puzzle-игра от Tactile Entertainment (жанр «connect three»).

По мере прохождения уровней игрок сталкивается с __гейтами__ — точками, где его останавливают и заставляют либо подождать, либо сделать внутриигровую покупку, чтобы продолжить.

Изначально первый гейт стоял на __30-м уровне__.

Команда провела A/B-тест, в котором первый гейт перенесли на __40-й уровень__, и хотела понять, как это повлияет на удержание игроков.
* Control: gate_30
* Treatment: gate_40

Данные: 90 189 игроков, которые установили игру во время эксперимента.

Каждый игрок случайно попадал в одну из двух групп.

Метрики, которые собирали:
* retention_1 — вернулся ли игрок на следующий день после установки
* retention_7 — вернулся ли на 7-й день
* sum_gamerounds — сколько раундов сыграл за первые 14 дней

# Импорт библиотек и чтение данных

In [46]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chisquare, binomtest, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [22]:
df = pd.read_csv('tables/cookie_cats.csv')
df.sample(5)

,userid,version,sum_gamerounds,retention_1,retention_7
39681,4394484,gate_40,36,True,False
13879,1538559,gate_30,161,True,True
42372,4687410,gate_40,0,False,False
44668,4947049,gate_30,186,True,True
10542,1162565,gate_30,1,False,False


# Обработка данных

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB


Пропусков нет.

In [24]:
df['userid'].duplicated().sum()

np.int64(0)

Дубликатов по userid нет.

In [25]:
df[df['sum_gamerounds'] < 0]['sum_gamerounds'].count()

np.int64(0)

Нет отрицательных чисел в `sum_gamerounds`.

In [26]:
df['version'].unique()

<StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

В колонке `version` только нужные нам значения.

In [27]:
df[df['version'] == 'gate_30']['sum_gamerounds'].describe()

count    44700.000000
mean        52.456264
std        256.716423
min          0.000000
25%          5.000000
50%         17.000000
75%         50.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64

In [28]:
df[df['version'] == 'gate_40']['sum_gamerounds'].describe()

count    45489.000000
mean        51.298776
std        103.294416
min          0.000000
25%          5.000000
50%         16.000000
75%         52.000000
max       2640.000000
Name: sum_gamerounds, dtype: float64

Работа с `sum_gamerounds`

Распределение количества раундов сильно скошено вправо (есть экстремальный выброс 49 854).

Поэтому:
* среднее значение приводим для справки,
* основные выводы и статистические тесты делаем по медиане и с помощью непараметрического критерия (Mann-Whitney U).

Игроков с нулевым количеством раундов не удаляем — это важная часть поведения пользователей.

# Проверка соотношение групп (SRM)

In [29]:
print('Абсолютные значения:\n', df['version'].value_counts().sort_index(), sep='', end='\n')
print('Доли:\n', df['version'].value_counts(normalize=True).sort_index() * 100, sep='', end='\n')
print(
    'Разница пользователей: ',
    abs(df['version'].value_counts().loc['gate_40'] - df['version'].value_counts().loc['gate_30'])
)

Абсолютные значения:
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
Доли:
version
gate_30    49.562585
gate_40    50.437415
Name: proportion, dtype: float64
Разница пользователей:  789


Группы распределены не равномерно, посмотрим есть ли srm.

In [30]:
# chi-square
observed = df['version'].value_counts().sort_index()
expected = [len(df) / 2, len(df) / 2]

chi2, p_chi = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square SRM test")
print(f"Observed: {observed.to_dict()}")
print(f"Expected: {expected}")
print(f"chi2 = {chi2:.3f}, p-value = {p_chi:.4f}")

# binomial test
n_gate40 = (df['version'] == 'gate_40').sum()
n_total = len(df)
binom_result = binomtest(n_gate40, n_total, p=0.5)
print(f"Binomial test p-value = {binom_result.pvalue:.4f}")

Chi-square SRM test
Observed: {'gate_30': 44700, 'gate_40': 45489}
Expected: [45094.5, 45094.5]
chi2 = 6.902, p-value = 0.0086
Binomial test p-value = 0.0087


Для проверки Sample Ratio Mismatch (SRM) я использую базовый уровень значимости **α = 0.05**, а также проверяю более строгий порог **α = 0.01**. В индустриальной практике для SRM часто применяют сверхстрогий порог (**α = 0.001**), так как это диагностический тест валидности сплитования, позволяющий отсеять случайные колебания на больших выборках и не браковать корректные эксперименты.

В нашем случае получено **p-value = 0.0086** (соотношение групп **49.56% / 50.44%**, разница — 789 пользователей):
* при **α = 0.05** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.01** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.001** — дисбаланс не является статистически значимым (тест пройден по индустриальному стандарту).

**Вывод и дальнейшие шаги:** Фиксирую наличие лёгкого дисбаланса. Поскольку по строгому индустриальному порогу (α = 0.001) тест проходит проверку, я продолжаю анализ. Однако все финальные продуктовые выводы будут сопровождаться дисклеймером о возможном смещении выборки.


# Тест гипотез

## Анализ удержания на 7-й день

In [31]:
# Расчет z-теста для двух пропорций
control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

success = [control.sum(), treatment.sum()]
n = [control.shape[0], treatment.shape[0]]

z_stat, p_value = proportions_ztest(count=success, nobs=n, alternative='two-sided')

cr_control = control.mean()
cr_treatment = treatment.mean()
diff = cr_treatment - cr_control

print(f"Control retention_7:   {cr_control:.4%}")
print(f"Treatment retention_7: {cr_treatment:.4%}")
print(f"Absolute difference:   {diff:.4%}")
print(f"Z-statistic:           {z_stat:.4f}")
print(f"P-value:               {p_value:.4f}")

Control retention_7:   19.0201%
Treatment retention_7: 18.2000%
Absolute difference:   -0.8201%
Z-statistic:           3.1644
P-value:               0.0016


In [32]:
# Доверительный интервал для двух пропорций
ci_low, ci_upp = confint_proportions_2indep(
    count1=treatment.sum(),
    nobs1=treatment.shape[0],
    count2=control.sum(),
    nobs2=control.shape[0],
    compare='diff',
    alpha=0.05
)
print(f"95% CI for difference (treatment - control): [{ci_low:.4%}, {ci_upp:.4%}]")

95% CI for difference (treatment - control): [-1.3282%, -0.3121%]


In [38]:
# bootstrap доверительный интервал
np.random.seed(42)
n_boot = 10000

boot_diff = []
for _ in range(n_boot):
    sample_control = np.random.choice(control, size=len(control), replace=True)
    sample_treatment = np.random.choice(treatment, size=len(treatment), replace=True)
    boot_diff.append(sample_treatment.mean() - sample_control.mean())

boot_diff = np.array(boot_diff)

ci_low, ci_upp = np.percentile(boot_diff, q=[2.5, 97.5])
print(f'Bootstrap 95% CI for difference (treatment - control): [{ci_low:.4%}, {ci_upp:.4%}]')

Bootstrap 95% CI for difference (treatment - control): [-1.3169%, -0.3155%]


## Анализ удержания на 1-й день

In [39]:
# Расчет z-теста для двух пропорций
control = df[df['version'] == 'gate_30']['retention_1']
treatment = df[df['version'] == 'gate_40']['retention_1']

success = [control.sum(), treatment.sum()]
n = [control.shape[0], treatment.shape[0]]

z_stat, p_value = proportions_ztest(count=success, nobs=n, alternative='two-sided')

cr_control = control.mean()
cr_treatment = treatment.mean()
diff = cr_treatment - cr_control

print(f'Control retention_1: {cr_control:.4%}')
print(f'Treatment retention_1: {cr_treatment:.4%}')
print(f'Absolute difference: {diff:.4%}')
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value: {p_value:.4f}')

Control retention_1: 44.8188%
Treatment retention_1: 44.2283%
Absolute difference: -0.5905%
Z-statistic: 1.7841
P-value: 0.0744


In [43]:
# Доверительный интервал для двух пропорций
ci_low, ci_upp = confint_proportions_2indep(
    count1=treatment.sum(),
    nobs1=treatment.shape[0],
    count2=control.sum(),
    nobs2=control.shape[0],
    compare='diff',
    alpha=0.05
)
print(f'95% CI for difference (treatment - control): [{ci_low:.4%}, {ci_upp:.4%}])')

95% CI for difference (treatment - control): [-1.2392%, 0.0582%])


In [41]:
# bootstrap доверительный интервал
np.random.seed(42)
n_boot = 10000

boot_diff = []
for _ in range(n_boot):
    sample_control = np.random.choice(control, size=len(control), replace=True)
    sample_treatment = np.random.choice(treatment, size=len(treatment), replace=True)
    boot_diff.append(sample_treatment.mean() - sample_control.mean())

boot_diff = np.array(boot_diff)

ci_low, ci_upp = np.percentile(boot_diff, q=[2.5, 97.5])
print(f'Bootstrap 95% CI for difference (treatment - control): [{ci_low:.4%}, {ci_upp:.4%}]')

Bootstrap 95% CI for difference (treatment - control): [-1.2366%, 0.0586%]


## Анализ количества раундов

In [49]:
# Расчет теста Мана Уитни
control = df[df['version'] == 'gate_30']['sum_gamerounds']
treatment = df[df['version'] == 'gate_40']['sum_gamerounds']

# Описательные статистики
print("Control:")
print(f"  Mean:   {control.mean():.2f}")
print(f"  Median: {control.median():.1f}")
print("Treatment:")
print(f"  Mean:   {treatment.mean():.2f}")
print(f"  Median: {treatment.median():.1f}")
print()

u_stat, p_value = mannwhitneyu(control, treatment, alternative='two-sided')

print(f'Mann-Whitney U statistic: {u_stat:.1f}')
print(f'P-value: {p_value:.4f}')

Control:
  Mean:   52.46
  Median: 17.0
Treatment:
  Mean:   51.30
  Median: 16.0

Mann-Whitney U statistic: 1024331250.5
P-value: 0.0502


# Выводы и рекомендации

__Что показали данные__

* Перенос первого гейта с 30-го на 40-й уровень __статистически значимо снизил 7-дневное удержание__ на 0.82 процентных пункта (p = 0.0016). 95% доверительный интервал: [−1.33%, −0.31%]. \
Ноль в интервал не входит — эффект отрицательный и устойчивый.

* По day-1 retention значимой разницы нет (p = 0.074). \
Наблюдается небольшое снижение (−0.59 п.п.), но оно может быть случайным.

* По количеству сыгранных раундов значимой разницы тоже нет (Mann-Whitney p = 0.050). \
Медиана отличается всего на 1 раунд, что практически незаметно.

__Рекомендация__

Не переносить гейт на 40-й уровень. Оставляем гейт на 30-м уровне: это сохраняет более высокое 7-дневное удержание, а краткосрочное удержание и вовлечённость при этом не выигрывают от переноса.

# «Как бы я спроектировал тест, если бы запускал сам»